### 🧠 Part A — Data Merging and Reporting Pipeline
Key Concepts Used

pathlib.glob() → find files

csv.DictReader() → read CSV as dictionary

set() → remove duplicates

json.dump() → export JSON

datetime → metadata timestamp

In [ ]:
# import required libraries
import csv
import json
from pathlib import Path
from datetime import datetime

# folder containing csv files
data_folder = Path(".")

# find all csv files like data1.csv, data2.csv, data3.csv
csv_files = list(data_folder.glob("data*.csv"))

all_rows = []

# read all csv files
for file in csv_files:
    with open(file, "r", newline="") as f:
        reader = csv.DictReader(f)

        # add rows to master list
        for row in reader:
            all_rows.append(row)

# -------------------------------------
# REMOVE DUPLICATES
# -------------------------------------

unique_rows = []
seen = set()

for row in all_rows:
    key = (row["date"], row["product"], row["qty"], row["price"])

    if key not in seen:
        seen.add(key)
        unique_rows.append(row)

# -------------------------------------
# SORT BY DATE
# -------------------------------------

unique_rows.sort(key=lambda x: x["date"])

# -------------------------------------
# CALCULATE REVENUE PER PRODUCT
# -------------------------------------

revenue_by_product = {}

for row in unique_rows:
    product = row["product"]
    qty = int(row["qty"])
    price = float(row["price"])

    revenue = qty * price

    if product not in revenue_by_product:
        revenue_by_product[product] = 0

    revenue_by_product[product] += revenue

# -------------------------------------
# EXPORT MERGED CSV
# -------------------------------------

with open("merged_sales.csv", "w", newline="") as f:
    fieldnames = ["date", "product", "qty", "price"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(unique_rows)

# -------------------------------------
# JSON METADATA
# -------------------------------------

total_revenue = sum(revenue_by_product.values())

output = {
    "metadata": {
        "files_processed": len(csv_files),
        "total_rows": len(unique_rows),
        "total_revenue": total_revenue,
        "generated_at": datetime.now().isoformat()
    },
    "revenue_by_product": revenue_by_product
}

# write json
with open("revenue_summary.json", "w") as f:
    json.dump(output, f, indent=4)

### 🚀 Part B — Backup Manager with Rotation

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime
import sys

# command line arguments
source_directory = Path(sys.argv[1])
backup_directory = Path(sys.argv[2])

backup_directory.mkdir(exist_ok=True)

log_file = "backup_log.txt"

# allowed extensions
extensions = [".csv", ".json"]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for file in source_directory.iterdir():

    if file.suffix in extensions:

        new_name = f"{file.stem}_{timestamp}{file.suffix}"
        destination = backup_directory / new_name

        # copy file
        shutil.copy2(file, destination)

        # log operation
        with open(log_file, "a") as log:
            log.write(f"Copied {file} -> {destination}\n")

        # -------------------------------
        # BACKUP ROTATION
        # -------------------------------

        backups = sorted(
            backup_directory.glob(f"{file.stem}_*{file.suffix}"),
            key=lambda x: x.stat().st_mtime
        )

        while len(backups) > 5:
            old_file = backups.pop(0)
            old_file.unlink()

            with open(log_file, "a") as log:
                log.write(f"Deleted old backup {old_file}\n")

### 💼 Part C — Interview Ready

### Q1 — json.load vs json.loads
json.load()

Definition

json.load() reads JSON data directly from a file object and converts it into a Python dictionary.

Usage

Used when JSON is stored inside a file.

In [ ]:
import json

with open("data.json", "r") as f:
    data = json.load(f)

print(data)

###
json.loads()

Definition

json.loads() converts a JSON formatted string into a Python dictionary.

Usage

Used when JSON comes from an API or string variable.

### Q2 — Find Large Files


In [ ]:
from pathlib import Path

def find_large_files(directory, size_mb):

    directory = Path(directory)

    results = []

    for file in directory.rglob("*"):

        if file.is_file():

            size = file.stat().st_size / (1024 * 1024)

            if size > size_mb:
                results.append((file.name, round(size, 2)))

    # sort descending
    results.sort(key=lambda x: x[1], reverse=True)

    return results

### Q3 — Fixed Code (Debug)

In [ ]:
import csv

def merge_csv_files(file_list):

    all_data = []
    header_saved = False

    for filename in file_list:

        with open(filename, "r", newline="") as f:
            reader = csv.reader(f)

            header = next(reader)

            # write header only once
            if not header_saved:
                all_data.append(header)
                header_saved = True

            for row in reader:
                all_data.append(row)

    with open("merged.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerows(all_data)

    return len(all_data)

### Fixes:

1️⃣ Added import csv
2️⃣ Added newline="" to avoid blank lines
3️⃣ Prevented duplicate headers

### 🤖 Part D — AI Augmented Task

### Write a Python script that reads a CSV file and automatically detects the delimiter using Python's csv module.

Possible delimiters:
comma
tab
semicolon
pipe

The script should convert the CSV into JSON format.

Requirements:
- Use csv.Sniffer() to detect delimiter
- Read CSV safely
- Export formatted JSON file

### 
import csv
import json

def csv_to_json(input_file, output_file):

    with open(input_file, "r") as f:

        sample = f.read(1024)
        f.seek(0)

        dialect = csv.Sniffer().sniff(sample)

        reader = csv.DictReader(f, dialect=dialect)

        data = list(reader)

    with open(output_file, "w") as f:
        json.dump(data, f, indent=4)

csv_to_json("data.csv", "output.json")

### 200-Word Evaluation

The AI-generated script successfully implemented automatic delimiter detection using Python's csv.Sniffer() class. This is an appropriate and widely recommended approach for handling CSV files with unknown delimiters. The code correctly reads a sample of the file to infer the delimiter and then resets the file pointer using seek(0) before parsing the data. It also uses csv.DictReader, which is a good design choice because it converts rows directly into dictionaries, making JSON conversion straightforward.

Another positive aspect of the script is its simplicity and readability. The use of json.dump() with indent=4 ensures that the resulting JSON file is human-readable and well structured. For typical CSV files with headers and standard formatting, the script should work reliably.

However, the AI-generated solution does not fully handle several edge cases. For example, it assumes the CSV file always contains a header row, which may not always be true. Additionally, it does not include error handling for malformed files, encoding issues, or cases where the delimiter cannot be detected. It also does not validate input file paths or handle very large CSV files efficiently.

An improvement would be to add exception handling, encoding specification (utf-8), and fallback delimiters if csv.Sniffer() fails.